# MVP sobre datos reales (anonimizados) — Mesa de servicio IX

Usa el esquema real: **`issues`** (Jira: Resumen, Estado, Prioridad, Creada...) para sentimiento/riesgo y Jira; **`ingresadas`** (bti_clasificacion, resumen_problema, accion_ops, fecha_orden) para Causas BTI y Operacional.

> Requiere haber re-generado la base con el `anonimizar_jira.py` corregido (que enmascara `Persona asignada`, `Descripción`, `Comentario` y los números de identificación). Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Drive + localizar la base (la más grande)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os
FILE_ID = '1FZhxpcYeKsriLIHpKSJlfZNjygXZXqmX'
cands = glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True)
for c in cands: print(round(os.path.getsize(c)/1e6,1),'MB ->',c)
if cands:
    RUTA_DB = max(cands, key=os.path.getsize); print('USANDO:', RUTA_DB)
else:
    import gdown; RUTA_DB='/content/incidentes_anonimizado.db'; gdown.download(id=FILE_ID, output=RUTA_DB, quiet=False)

## 2. Cargar `issues` (Jira) y utilidades

In [ ]:
import sqlite3, pandas as pd, numpy as np, re, json
from datetime import datetime
con = sqlite3.connect(RUTA_DB)
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)['name'].tolist()
print('Tablas:', tablas)

def load(nombre, cols='*'):
    if nombre in tablas:
        t = pd.read_sql('SELECT %s FROM "%s"' % (cols, nombre), con)
        t.columns = t.columns.str.lower().str.strip()
        return t
    return None

df = load('issues')
print('issues:', df.shape)
print('Columnas:', df.columns.tolist())
df.head(3)

## 3. Sentimiento / frustración sobre `Resumen`

In [ ]:
COL_TEXTO = 'resumen' if 'resumen' in df.columns else df.select_dtypes('object').columns[0]
df = df.drop_duplicates()
df[COL_TEXTO] = df[COL_TEXTO].astype(str).str.strip()

POS = set('gracias ok correcto solucionado resuelto disponible exitoso'.split())
NEG = set('error falla fallo sin no cancelacion cancelación pendiente caido caída bloqueo rechazo duplicado incidencia problema demora retraso urgente'.split())
def sent(t):
    pal = re.findall(r'[a-záéíóúñ]+', str(t).lower())
    p = sum(w in POS for w in pal); n = sum(w in NEG for w in pal)
    return round((p-n)/(p+n),3) if (p+n) else 0.0
df['sentimiento'] = df[COL_TEXTO].apply(sent)
df['frustracion'] = ((1-df['sentimiento'])/2*100).round(1)
df['banda'] = df['frustracion'].apply(lambda f:'alto' if f>=66 else ('medio' if f>=40 else 'bajo'))
df[['sentimiento','frustracion']].describe()

## 4. Tendencia mensual (columna `Creada`)

In [ ]:
COL_FECHA = next((c for c in ['creada','fecha_creacion','fecha','created','actualizada'] if c in df.columns), None)
serie_mensual = {}
if COL_FECHA:
    fechas = pd.to_datetime(df[COL_FECHA], errors='coerce', utc=True)
    serie = fechas.dt.to_period('M').astype(str).value_counts()
    serie = serie[serie.index != 'NaT'].sort_index()
    serie_mensual = {str(k): int(v) for k, v in serie.items()}
print('Tendencia mensual:', serie_mensual)

## 5. Bloques: KPIs, Jira, Causas BTI, Operacional

In [ ]:
# Clasificacion para el resumen: prioridad si existe, si no estado
COL_LABEL = next((c for c in ['prioridad','tipo de incidencia','estado'] if c in df.columns), None)
top_cat = df[COL_LABEL].value_counts().head(8).to_dict() if COL_LABEL else {}
bins=[-1,-0.6,-0.2,0.2,0.6,1.0001]; etq=['muy negativo','negativo','neutro','positivo','muy positivo']
hist=pd.cut(df['sentimiento'],bins=bins,labels=etq,include_lowest=True).value_counts().reindex(etq).fillna(0).astype(int)
alta=df[df['banda']=='alto']
impulsores=([{'variable':str(k)[:40],'importancia':round(float(v),3)} for k,v in alta[COL_LABEL].value_counts(normalize=True).head(6).items()] if (COL_LABEL and len(alta)) else [])

# Jira (desde issues)
idc = 'clave' if 'clave' in df.columns else df.columns[0]
est = 'estado' if 'estado' in df.columns else None
jira = {'total': int(len(df))}
if est: jira['por_estado'] = {str(k):int(v) for k,v in df[est].value_counts().head(8).items()}
jira['recientes'] = [{'id':str(r[idc])[:18],'resumen':str(r[COL_TEXTO])[:80],'estado':(str(r[est]) if est else '')} for _,r in df.head(15).iterrows()]

def recom(f): return 'Contacto inmediato y plan de retencion' if f>=66 else ('Seguimiento proactivo' if f>=40 else 'Monitoreo estandar')
tickets=[{'id':str(r[idc])[:18],'cliente':str(r[idc])[:18],'clasificacion':(str(r[COL_LABEL])[:30] if COL_LABEL else ''),'frustracion':float(r['frustracion']),'riesgo':float(r['frustracion']),'recomendacion':recom(r['frustracion'])} for _,r in df.sort_values('frustracion',ascending=False).head(15).iterrows()]

# Causas BTI (desde ingresadas)
bti_tabla=[]; operacional={}
ing = load('ingresadas')
if ing is not None:
    code = next((c for c in ['bti_clasificacion','clasificacion','bti'] if c in ing.columns), None)
    desc = next((c for c in ['resumen_problema','descripcion','detalle'] if c in ing.columns), None)
    if code:
        conteo = ing[code].value_counts().head(15)
        dmap = {}
        if desc:
            for _,row in ing.iterrows():
                k=str(row[code]); dmap.setdefault(k, str(row[desc]))
        bti_tabla=[{'codigo':str(k),'descripcion':dmap.get(str(k),'')[:90],'casos':int(v)} for k,v in conteo.items()]
    aco = next((c for c in ['accion_ops','accion','estado'] if c in ing.columns), None)
    operacional['ingresadas_total']=int(len(ing))
    if aco: operacional['por_accion']={str(k):int(v) for k,v in ing[aco].value_counts().head(8).items()}
ordn = load('ordenes_diarias')
if ordn is not None: operacional['ordenes_total']=int(len(ordn))

resultados={
 'generado': datetime.now().strftime('%Y-%m-%d %H:%M')+' (datos reales IX anonimizados)',
 'kpis':{'total_tickets':int(len(df)),'phishing_aislados':0,'procesados':int(len(df)),'pct_alto_riesgo':round(float((df['banda']=='alto').mean()*100),1),'frustracion_promedio':round(float(df['frustracion'].mean()),1),'sentimiento_promedio':round(float(df['sentimiento'].mean()),3)},
 'clasificacion':{str(k)[:30]:int(v) for k,v in top_cat.items()},
 'distribucion_riesgo':{k:int(v) for k,v in df['banda'].value_counts().reindex(['bajo','medio','alto']).fillna(0).astype(int).items()},
 'sentimiento_hist':{'bins':etq,'conteo':hist.tolist()},
 'impulsores':impulsores,'serie_mensual':serie_mensual,
 'jira':jira,'bti_tabla':bti_tabla,'operacional':operacional,'tickets_alto_riesgo':tickets}

with open('resultados.json','w',encoding='utf-8') as f:
    json.dump(resultados,f,ensure_ascii=False,indent=2)
print('KPIs:', resultados['kpis'])
print('Jira:', jira.get('total'),'| BTI:',len(bti_tabla),'| Operacional:',operacional)
print('resultados.json listo.')

## RUTA A — Descargar `resultados.json`

In [ ]:
try:
    from google.colab import files; files.download('resultados.json')
except Exception as e:
    print('Descarga manual. Detalle:', e)

## RUTA B — Publicar directo al dashboard (GitHub API)
Token fino con permiso Contents: Read and write sobre `mesa-servicio-ia`.

In [ ]:
import base64, requests, getpass
GH_OWNER='ricardocasallas3-ux'; GH_REPO='mesa-servicio-ia'; GH_PATH='docs/data/resultados.json'; GH_BRANCH='main'
token = getpass.getpass('Pega tu token de GitHub (no se mostrara): ')
headers={'Authorization':'token '+token,'Accept':'application/vnd.github+json'}
url='https://api.github.com/repos/%s/%s/contents/%s' % (GH_OWNER,GH_REPO,GH_PATH)
r=requests.get(url,headers=headers,params={'ref':GH_BRANCH}); sha=r.json().get('sha') if r.status_code==200 else None
with open('resultados.json','rb') as f: contenido=base64.b64encode(f.read()).decode()
payload={'message':'Actualizar dashboard desde Colab','content':contenido,'branch':GH_BRANCH}
if sha: payload['sha']=sha
resp=requests.put(url,headers=headers,json=payload)
print(resp.status_code, 'https://%s.github.io/%s/' % (GH_OWNER,GH_REPO) if resp.status_code in (200,201) else resp.json())